In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

df = pd.read_csv("../data/cleaned/cleaned_credit_default.csv")

df.shape

(149016, 19)

In [2]:
y = df["SeriousDlqin2yrs"]
X = df.drop(columns=["SeriousDlqin2yrs"])

In [3]:
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ["AgeGroup"]
numeric_cols = X.columns.difference(categorical_cols).tolist()

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_cat = encoder.fit_transform(X[categorical_cols])
X_num = X[numeric_cols].to_numpy()

X_final = np.hstack([X_num, X_cat])

X_final.shape

(149016, 24)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Baseline Stats of all Algorithms

In [7]:
baseline_results = pd.read_csv(
    "../reports/baseline_results.csv"
)
baseline_results


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.937055,0.598394,0.150963,0.241100,0.824983
1,Decision Tree,0.898940,0.259053,0.282675,0.270349,0.612851
2,Random Forest,0.935076,0.527738,0.187943,0.277176,0.831068


### Baseline Results

The baseline models provide a reference point for evaluating improvements
from tree-based model development and hyperparameter tuning.

Random Forest currently provides the best baseline ROC-AUC and F1 Score,
while Decision Tree provides the highest recall.

## Starting with Tuning Hyperparameters

### 1. Decision Tree

In [9]:
from sklearn.tree import DecisionTreeClassifier

In [12]:
depths = [3 , 5 , 7 , 10 , 20 , None]

dt_depth_results = []

for depth in depths:

    dt = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    dt.fit(X_train , y_train)

    y_pred = dt.predict(X_test)
    y_prob = dt.predict_proba(X_test)[: , 1]

    dt_depth_results.append({
        "max_depth": depth,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

dt_depth_results = pd.DataFrame(dt_depth_results)

dt_depth_results

,max_depth,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,3.0,0.936519,0.576208,0.157042,0.246815,0.821844
1,5.0,0.936418,0.567753,0.167680,0.258897,0.844323
2,7.0,0.936351,0.558245,0.186930,0.280076,0.844958
3,10.0,0.933767,0.500000,0.196555,0.282182,0.811388
4,20.0,0.914575,0.317136,0.251266,0.280384,0.616142
5,NaN,0.898940,0.259053,0.282675,0.270349,0.612851


### `max_depth = 7`

1. **Best ROC-AUC:** `max_depth=7` → **0.8450**
2. **Best F1 Score:** `max_depth=10` → **0.2822**
3. **Best Recall:** `max_depth=None` → **0.2827**

In [16]:
min_split_values = [2 , 5 , 10 , 20 , 50 , 100]

dt_split_results = []

for min_split in min_split_values:

    dt = DecisionTreeClassifier(
        max_depth = 7,
        min_samples_split = min_split,
        random_state = 42
    )

    dt.fit(X_train , y_train)

    y_pred = dt.predict(X_test)
    y_prob = dt.predict_proba(X_test)[: , 1]

    dt_split_results.append({
        "min_samples_split": min_split,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

dt_split_results = pd.DataFrame(dt_split_results)

dt_split_results

,min_samples_split,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,2,0.936351,0.558245,0.186930,0.280076,0.844958
1,5,0.936351,0.558245,0.186930,0.280076,0.844895
2,10,0.936351,0.558245,0.186930,0.280076,0.844893
3,20,0.936351,0.558245,0.186930,0.280076,0.844910
4,50,0.936217,0.555053,0.186424,0.279105,0.844862
5,100,0.936183,0.554381,0.185917,0.278452,0.844950


### min_samples_split

With `max_depth=7` fixed, changing `min_samples_split` from 2 to 100
produced almost no change in model performance.

The ROC-AUC remained around 0.845 and the F1 Score remained around 0.28.
Therefore, `min_samples_split` has little effect on this Decision Tree
within the tested range.

In [17]:
min_leaf_values = [1, 2, 5, 10, 20, 50, 100]

dt_leaf_results = []

for min_leaf in min_leaf_values:

    dt = DecisionTreeClassifier(
        max_depth=7,
        min_samples_split=2,
        min_samples_leaf=min_leaf,
        random_state=42
    )

    dt.fit(X_train, y_train)

    y_pred = dt.predict(X_test)
    y_prob = dt.predict_proba(X_test)[:, 1]

    dt_leaf_results.append({
        "min_samples_leaf": min_leaf,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

dt_leaf_results = pd.DataFrame(dt_leaf_results)

dt_leaf_results

,min_samples_leaf,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,1,0.936351,0.558245,0.186930,0.280076,0.844958
1,2,0.936384,0.559091,0.186930,0.280182,0.844858
2,5,0.936250,0.559871,0.175279,0.266975,0.846862
3,10,0.935881,0.552239,0.168693,0.258440,0.846828
4,20,0.935478,0.541599,0.168186,0.256668,0.846102
5,50,0.936082,0.560209,0.162614,0.252061,0.849752
6,100,0.936451,0.575188,0.155015,0.244214,0.850897


### `min_samples_leaf = 2`

1. **Best ROC-AUC:** `min_samples_leaf=100` → **0.8509**
2. **Best F1:** `min_samples_leaf=2` → **0.2802**
3. **Best Recall:** `min_samples_leaf=2` → **0.1869**

### min_samples_leaf

Increasing `min_samples_leaf` improved ROC-AUC but reduced recall and F1.
The highest ROC-AUC of 0.8509 was obtained with `min_samples_leaf=100`,
while the highest F1 Score of 0.2802 was obtained with `min_samples_leaf=2`.

Therefore, the choice of `min_samples_leaf` depends on the evaluation metric.

In [18]:
class_weights = [
    None,
    "balanced",
    {0: 1, 1: 2},
    {0: 1, 1: 3},
    {0: 1, 1: 5}
]

dt_weight_results = []

for weight in class_weights:

    dt = DecisionTreeClassifier(
        max_depth=7,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight=weight,
        random_state=42
    )

    dt.fit(X_train, y_train)

    y_pred = dt.predict(X_test)
    y_prob = dt.predict_proba(X_test)[:, 1]

    dt_weight_results.append({
        "class_weight": str(weight),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

dt_weight_results = pd.DataFrame(dt_weight_results)

dt_weight_results

,class_weight,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,None,0.936351,0.558245,0.186930,0.280076,0.844958
1,balanced,0.776507,0.196791,0.770517,0.313511,0.845032
2,"{0: 1, 1: 2}",0.926621,0.434300,0.356636,0.391655,0.845100
3,"{0: 1, 1: 3}",0.921521,0.408245,0.411348,0.409791,0.841825
4,"{0: 1, 1: 5}",0.893169,0.323099,0.559777,0.409714,0.844719


### class_weight

Class weighting had a significant effect on Decision Tree performance.

Increasing the weight of the default class improved recall substantially,
but reduced precision. The `balanced` setting achieved the highest recall
of 0.7705, but its precision decreased to 0.1968.

The custom weight `{0:1, 1:3}` achieved the highest F1 Score of 0.4098,
while `{0:1, 1:5}` provided similar F1 with higher recall.

This demonstrates that class weighting is an important hyperparameter
for this imbalanced classification problem.

### `class_weight = {0: 1, 1: 3}`

1. **Precision:** 0.4082
2. **Recall:** 0.4113
3. **F1 Score:** **0.4098**
4. **ROC-AUC:** 0.8418

In [19]:
ccp_values = [0, 0.000001, 0.000005, 0.00001, 0.00005, 0.0001]

dt_pruning_results = []

for alpha in ccp_values:

    dt = DecisionTreeClassifier(
        max_depth=7,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight={0: 1, 1: 3},
        ccp_alpha=alpha,
        random_state=42
    )

    dt.fit(X_train, y_train)

    y_pred = dt.predict(X_test)
    y_prob = dt.predict_proba(X_test)[:, 1]

    dt_pruning_results.append({
        "ccp_alpha": alpha,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

dt_pruning_results = pd.DataFrame(dt_pruning_results)

dt_pruning_results

,ccp_alpha,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,0.000000,0.921521,0.408245,0.411348,0.409791,0.841825
1,0.000001,0.921521,0.408245,0.411348,0.409791,0.841825
2,0.000005,0.921521,0.408245,0.411348,0.409791,0.841825
3,0.000010,0.921521,0.408245,0.411348,0.409791,0.842263
4,0.000050,0.921051,0.408410,0.428065,0.418006,0.846369
5,0.000100,0.920749,0.407882,0.435157,0.421078,0.847603


### ccp_alpha — Cost-Complexity Pruning

Cost-complexity pruning produced only a modest change in performance.
Increasing `ccp_alpha` to 0.0001 improved F1 Score from 0.4098 to 0.4211
and ROC-AUC from 0.8418 to 0.8476.

Therefore, pruning provides a small improvement in this experiment, but
there is no strong evidence that pruning is solving a major overfitting
problem in the current Decision Tree configuration.

### `ccp_alpha = 0.0001`

1. **Precision:** 0.4079
2. **Recall:** 0.4352
3. **F1 Score:** **0.4211**
4. **ROC-AUC:** 0.8476